# Chapter 2: Lab 1 - Hands-on lab: Retrieving fundamental ratios for Apple

In this hands-on lab, we implement an agent using the **tool use pattern**. This agent will be requested to retrieve the liquidity ratios of Apple.

We use OpenAI Agents SDK with GPT-4o-mini as the underlying LLM.

The tool used in this hands-on lab "`get_company_fundamentals`" will invoke Finance Toolkit that is using the Financial Modeling Prep API to collect various sets of fundamental ratios:

https://www.jeroenbouma.com/projects/financetoolkit

https://site.financialmodelingprep.com/

You need to create an API Key on https://site.financialmodelingprep.com/: create an account and then create an API Key visible in your dashboard.


## Install Packages

In [1]:
%pip install openai-agents -q
%pip install financetoolkit -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.4/169.4 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.7/365.7 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 31.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.5/63.5 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 108.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas=

Specify the API Keys:

In [3]:
from google.colab import userdata
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

import os
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

FINANCIAL_MODELING_PREP_API_KEY = userdata.get('FINANCIAL_MODELING_PREP_API_KEY')

In [4]:
from financetoolkit import Toolkit

In [5]:
from agents import Agent, Runner, WebSearchTool, trace, ItemHelpers, TResponseInputItem, function_tool
import pandas as pd

## Specify the tool

The tool `get_company_fundamentals` retrieves a stock’s fundamental ratios, including profitability, valuation, and liquidity metrics.


In [6]:
@function_tool
def get_company_fundamentals(ticker: str) -> dict[str, pd.DataFrame]:
  """ Get profitability, valuation and liquidity ratios for a given ticker. """

  companies = Toolkit(
      [ticker], api_key=FINANCIAL_MODELING_PREP_API_KEY, start_date="2022-01-01"
  )
  ratios = companies.ratios.collect_all_ratios()

  # Keep only profitability, valuation and liquidity ratios
  profitability_ratios_names = ['Return on Equity','Return on Assets','Gross Margin','Operating Margin','Net Profit Margin']
  valuation_ratios_names = ['Price-to-Earnings',  "Price-to-Book", "Price-to-Cash-Flow", "Price-to-Free-Cash-Flow", 'EV-to-EBITDA','EV-to-Sales']
  liquidity_ratios_names = ["Current Ratio","Quick Ratio","Cash Ratio","Working Capital","Operating Cash Flow Ratio","Operating Cash Flow to Sales Ratio","Short Term Coverage Ratio"]


  return {
      "profitability_ratios":ratios.loc[ratios.index.intersection(profitability_ratios_names)],
      "valuation_ratios":ratios.loc[ratios.index.intersection(valuation_ratios_names)],
      "liquidity_ratios": ratios.loc[ratios.index.intersection(liquidity_ratios_names)]}


This function defines how to extract key fundamental ratios for a given company ticker.

It connects to a financial data provider, collects all available financial ratios starting from 2022, and then filters them into three categories: profitability, valuation, and liquidity.

* The **profitability** ratios measure how efficiently a company generates profit relative to its revenue, assets, or equity. They indicate the firm’s ability to create value from its operations (e.g., Return on Equity (ROE), Net Profit Margin).  
* The **valuation** ratios assess how the market values the company compared to its earnings, cash flows, or assets. They help determine whether a stock appears expensive or undervalued (e.g., Price-to-Earnings (PE), EV-to-EBITDA).
* The **liquidity** ratios evaluate a company’s ability to meet its short-term obligations. They indicate whether the firm has sufficient liquid resources to cover immediate liabilities (e.g., Current Ratio (CR), Quick Ratio(QR)).

Instead of returning the full dataset, the function selectively returns only the most important ratios for our use case, such as Return on Equity (ROE), Price-to-Earnings (PE), and Current Ratio (CR), organized into structured output.

## Specify the Fundamental retrieval Agent

 The `fundamental_retrieval_agent` is instructed to invoke the fundamentals tool when needed and return the desired set of financial ratios for the specified ticker:


In [7]:
MODEL = "gpt-4o-mini"
fundamental_retrieval_agent = Agent(
    name="Fundamental Retrieval Agent",
    model=MODEL,
    instructions="""
You are a Fundamental Retrieval Agent.

Tasks:
1) Call the fundamentals tool for the specified ticker.
2) Output the desired set of ratios.

""",
    tools=[get_company_fundamentals],
)

## Run the agent

In the following prompt I ask the agent (LLM behind it) to retrieve ratios for the most recent fiscal year: (looks which year it retrieves ⬇ )

In [8]:

fundamental_results = await Runner.run(fundamental_retrieval_agent,
                                       """Proceed step by step:
                                       1- List 3 liquidity ratios and their signification.
                                       2- Provide Apple’s values for the three listed liquidity ratios for the most recent fiscal year.
                                       """)
print(fundamental_results.final_output)

2026-09-09 19:18:53 - financetoolkit - INFO - Obtaining financial statements
2026-09-09 19:18:53 - financetoolkit - INFO - Obtaining balance data for 1 ticker(s)
2026-09-09 19:18:53 - financetoolkit - INFO - Obtaining income data for 1 ticker(s)
2026-09-09 19:18:53 - financetoolkit - INFO - Obtaining cashflow data for 1 ticker(s)
2026-09-09 19:18:54 - financetoolkit - INFO - Obtaining treasury data for 1 ticker(s)
/usr/local/lib/python3.13/dist-packages/yfinance/scrapers/history.py:204: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
2026-09-09 19:18:55 - financetoolkit - INFO - Obtaining historical data for 2 ticker(s)


### Apple’s Values for the Three Listed Liquidity Ratios (Most Recent Fiscal Year - 2023)

1. **Current Ratio**
   - **Value**: 0.988
   - **Signification**: Indicates that Apple has slightly less current assets than current liabilities, but is close to a healthy liquidity position.

2. **Quick Ratio**
   - **Value**: 0.6267
   - **Signification**: Shows that when excluding inventory, Apple has about 62.67% of its current liabilities covered by its most liquid assets, which may indicate less liquidity.

3. **Cash Ratio**
   - **Value**: 0.4236
   - **Signification**: Suggests that Apple can cover approximately 42.36% of its short-term liabilities with cash and cash equivalents alone, suggesting a moderate level of liquidity.

If you have any further questions or need additional details, feel free to ask!


The agent retrieves data for 2023 because the LLM relies on its knowledge cutoff (2024) to infer what it considers the most recent completed year.

In this run, I modified slighlty the prompt to sepcify that we are in 2026:

In [9]:

fundamental_results = await Runner.run(fundamental_retrieval_agent,
                                       """Proceed step by step:
                                       1- List 3 liquidity ratios and explain their significance.
                                       2- Provide Apple’s values for the three listed liquidity ratios for the most recent fiscal year, knowing that we are in 2026.
                                       """)
print(fundamental_results.final_output)

2026-09-09 19:20:28 - financetoolkit - INFO - Obtaining financial statements
2026-09-09 19:20:28 - financetoolkit - INFO - Obtaining balance data for 1 ticker(s)
2026-09-09 19:20:29 - financetoolkit - INFO - Obtaining income data for 1 ticker(s)
2026-09-09 19:20:29 - financetoolkit - INFO - Obtaining cashflow data for 1 ticker(s)
2026-09-09 19:20:30 - financetoolkit - INFO - Obtaining treasury data for 1 ticker(s)
/usr/local/lib/python3.13/dist-packages/yfinance/scrapers/history.py:204: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
2026-09-09 19:20:30 - financetoolkit - INFO - Obtaining historical data for 2 ticker(s)


### Apple's Liquidity Ratios for the Most Recent Fiscal Year (2025)

1. **Current Ratio**: **0.8933**
   - This indicates that Apple has approximately 89 cents of current assets for every dollar of current liabilities, suggesting limited short-term liquidity.

2. **Quick Ratio**: **0.5704**
   - This means that Apple has about 57 cents of liquid assets (excluding inventory) available to cover each dollar of current liabilities, reflecting its ability to meet immediate obligations without relying on inventory sales.

3. **Cash Ratio**: **0.3302**
   - This indicates that Apple has 33 cents in cash and cash equivalents for every dollar of current liabilities, offering a conservative perspective on its ability to cover short-term obligations solely with cash.

### Summary
- **Current Ratio**: 0.8933
- **Quick Ratio**: 0.5704
- **Cash Ratio**: 0.3302

These ratios can provide insight into Apple's short-term financial health and liquidity position.


This time, the agent provides the correct information, it retrieves 2025 fiscal year data.

**Conclusion**:
This result illustrates the strength of the tool use pattern: the agent is able to retrieve up-to-date, structured financial data from an external source and present it in a clear, consistent format. It also provides insights for each ratio, along with an explanation of what is generally considered a healthy value.

Run the toolkit directly:

In [10]:
companies = Toolkit(
      ["AAPL"], api_key=FINANCIAL_MODELING_PREP_API_KEY, start_date="2022-01-01"
  )
ratios = companies.ratios.collect_all_ratios()

2026-09-09 19:21:16 - financetoolkit - INFO - Obtaining financial statements
2026-09-09 19:21:16 - financetoolkit - INFO - Obtaining balance data for 1 ticker(s)
2026-09-09 19:21:17 - financetoolkit - INFO - Obtaining income data for 1 ticker(s)
2026-09-09 19:21:17 - financetoolkit - INFO - Obtaining cashflow data for 1 ticker(s)
2026-09-09 19:21:18 - financetoolkit - INFO - Obtaining treasury data for 1 ticker(s)
/usr/local/lib/python3.13/dist-packages/yfinance/scrapers/history.py:204: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
2026-09-09 19:21:18 - financetoolkit - INFO - Obtaining historical data for 2 ticker(s)


In [11]:
liquidity_ratios_3 = [
    "Current Ratio",
    "Quick Ratio",
    "Cash Ratio",
]
ratios.loc[ratios.index.intersection(liquidity_ratios_3)]

,2022,2023,2024,2025
Current Ratio,0.8794,0.988,0.8673,0.8933
Quick Ratio,0.4967,0.6267,0.5589,0.5704
Cash Ratio,0.3137,0.4236,0.3695,0.3302
